In [ ]:
# https://www.youtube.com/watch?v=PaCmpygFfXo&list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ&index=2

In [309]:
import torch

In [310]:
words = open('names.txt').read().splitlines()

In [311]:
for i in range(len(words)):
    a = list(words[i])
    a.insert(0, '.')
    a.insert(len(a), '.')
    words[i] = ''.join(a)

In [312]:
all_letters = sorted(set(''.join(words)))
print(all_letters)

['.', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [313]:
N = torch.zeros((729, 27), dtype=torch.int32)

In [314]:
words[:3]

['.emma.', '.olivia.', '.ava.']

In [315]:
all_letters = sorted(set(''.join(words)))

In [316]:
stoi = {k: v for k,v in enumerate(all_letters)}
itos = {v: k for k, v in stoi.items()}

In [317]:
words[:10]

['.emma.',
 '.olivia.',
 '.ava.',
 '.isabella.',
 '.sophia.',
 '.charlotte.',
 '.mia.',
 '.amelia.',
 '.harper.',
 '.evelyn.']

In [318]:
dd = {}
i_count = 0
o_count = 0
for i in all_letters:
    o_count = o_count*27
    for j in all_letters:
        dd[i+j] = i_count + o_count
        i_count = i_count + 1

In [319]:
for word in words:
    for j in range(2, len(word)):
        N[dd[word[j-2] + word[j-1]], itos[word[j]]] += 1

In [320]:
(N == 0).sum()

tensor(13646)

In [321]:
729*27

19683

In [322]:
N = N + 1

In [323]:
N[48]

tensor([ 16,   1,  36,   1,  50,   2,   1,  28,   2,   3,   1,   4,  28,   5,
         67,   1,   1,   1, 109,  26,  14,   1,   1,   1,   6,   1,   1],
       dtype=torch.int32)

In [324]:
P = N / N.sum(1, keepdim=True)

In [325]:
P[48].exp().sum()

tensor(28.0738)

In [326]:
loss = []
n = 0
for word in words:
    for j in range(2, len(word)):
        row_exp_sum = torch.exp(P[dd[word[j-2] + word[j-1]]]).sum()
        p = torch.exp(P[dd[word[j-2] + word[j-1]], itos[word[j]]])
        l = -torch.log(p / row_exp_sum) 
        loss.append(l)
        n = n + 1
print(f"NLL {sum(loss) / n}")

NLL 3.143826723098755


In [327]:
P.shape

torch.Size([729, 27])

In [328]:
g = torch.Generator()

In [329]:
for _ in range(10):

    ix = 0
    generated_word = []
    generated_word.append('.')
    generated_word.append('.')

    while True:
        p = P[ix]
        c = torch.multinomial(p, num_samples = 1, replacement=True, generator=g)
    
        generated_word.append(stoi[c.item()])
        ix = dd[''.join(generated_word[-2:])]
    
        if stoi[c.item()] == "." and len(generated_word) > 3:
            break
    print(''.join(generated_word))

..triolaiwhetvia.
..al.
...wil.
...flon.
..helie.
..niveon.
..jedorgulivy.
..peanna.
..maie.
..fautvaylo.


## Neural Networks

In [330]:
for word in words[:1]:
    for j in range(2, len(word)):
        print(dd[word[j-2] + word[j-1]], itos[word[j]])

5 13
148 13
364 1
352 0


In [331]:
xs, ys = [], []
for word in words:
    for j in range(2, len(word)):
        xs.append(dd[word[j-2] + word[j-1]])
        ys.append(itos[word[j]])
xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [332]:
print(xs.shape)
print(ys.shape)

torch.Size([196113])
torch.Size([196113])


In [333]:
import torch.nn.functional as F

In [334]:
xenc = F.one_hot(xs, num_classes=729).float()

In [335]:
xenc.shape

torch.Size([196113, 729])

In [336]:
W = torch.randn((729, 27), requires_grad=True)

In [337]:
# xenc - 4,729
# W - 729, 27

z = (xenc @ W).exp()

In [338]:
z.shape

torch.Size([196113, 27])

In [339]:
P = z / z.sum(1, keepdim=True)

In [340]:
P.shape

torch.Size([196113, 27])

In [341]:
xenc = F.one_hot(xs, num_classes=729).float()
m = xenc.shape[0]
W = torch.randn((729, 27), requires_grad=True)

for i in range(200):
    # forward pass
    z = (xenc @ W).exp()
    P = z / z.sum(1, keepdim=True)
    
    # loss calculation
    loss = -torch.mean(torch.log(P[torch.arange(m), ys[torch.arange(m)]]))

    W.grad = None
    loss.backward()

    W.data = W.data - 150*W.grad

    if i % 20 == 0:
        print(f"Loss: {loss}")

Loss: 3.854048490524292
Loss: 2.5378594398498535
Loss: 2.3580925464630127
Loss: 2.280413866043091
Loss: 2.236145496368408
Loss: 2.2072598934173584
Loss: 2.1868813037872314
Loss: 2.1717374324798584
Loss: 2.1600446701049805
Loss: 2.150740623474121


In [342]:
for _ in range(10):

    ix = 0
    generated_word = []
    generated_word.append('.')
    generated_word.append('.')
    
    while True:
        
        
        xenc = F.one_hot(torch.tensor([ix]), num_classes=729).float()
        z = (xenc @ W).exp()
        P = z / z.sum(1, keepdim=True)
        
        c = torch.multinomial(P, num_samples = 1, replacement=True, generator=g)
    
        generated_word.append(stoi[c.item()])
        ix = dd[''.join(generated_word[-2:])]
    
        if stoi[c.item()] == "." and len(generated_word) > 3:
            break
    print(''.join(generated_word))

..ula.
..zicffnzylexlbtonwilvcbtvtwlqv.
..yandel.
..blay.
..vki.
..fart.
..halyn.
..elianean.
..uayn.
..ya.
